# Where in the funnel are columns lost?

*Setup cells below are carried from the shared analysis so this notebook runs on its own.*


In [9]:
import json
from pathlib import Path

import plotly.graph_objects as go

# Path to the materialized contract
contract_path = Path("../../references/feature-contract.json")

# Fallback structure if the contract hasn't been generated yet
if not contract_path.exists():
    print(f"⚠️ Warning: The contract file was not found at `{contract_path}`.")
    print("Run the `feature_contract` asset in Dagster to generate it first.")
    
else:
    with open(contract_path, "r") as f:
        contract_data = json.load(f)
        
    summary = contract_data.get("summary", {})
    fragments = contract_data.get("fragments", [])
    columns = contract_data.get("columns", [])

# Extract counts per check
rejections_by_check = {f["check"]: f["rejected"] for f in fragments}

In [7]:
from IPython.display import Markdown, display

table_md = f"""
| Metric | Count |
|--------|-------|
| **Declared Features** | {summary.get('declared', 0)} |
| **Admitted to Model** | {summary.get('admitted', 0)} |
| **Rejected by Audits** | {summary.get('rejected', 0)} |
| **Manual Overrides** | {summary.get('overridden', 0)} |
"""
display(Markdown(table_md))


| Metric | Count |
|--------|-------|
| **Declared Features** | 479 |
| **Admitted to Model** | 208 |
| **Rejected by Audits** | 271 |
| **Manual Overrides** | 0 |


## Feature Funnel (Waterfall)

The waterfall chart illustrates the "funnel" of our features. We start with the fully declared set, and then subtract the columns rejected by each sequential audit. 

In [8]:
# Prepare data for waterfall chart
steps = ["Declared"]
measures = ["absolute"]
values = [summary.get("declared", 0)]

for check, count in rejections_by_check.items():
    if count > 0:
        steps.append(f"Rejected: {check.replace('_', ' ').title()}")
        measures.append("relative")
        values.append(-count)

if summary.get("overridden", 0) > 0:
    steps.append("Manual Overrides")
    measures.append("relative")
    values.append(summary.get("overridden", 0))

steps.append("Admitted")
measures.append("total")
values.append(summary.get("admitted", 0))

fig = go.Figure(go.Waterfall(
    name="Feature Contract",
    orientation="v",
    measure=measures,
    x=steps,
    textposition="outside",
    text=[str(v) if m != "total" else str(v) for m, v in zip(measures, values)],
    y=values,
    connector={"line":{"color":"rgb(63, 63, 63)"}},
    decreasing={"marker":{"color":"#ef553b"}},
    increasing={"marker":{"color":"#00cc96"}},
    totals={"marker":{"color":"#636efa"}}
))

fig.update_layout(
    title="Feature Admittance Funnel",
    showlegend=False,
    plot_bgcolor='rgba(0,0,0,0)',
    yaxis_title="Number of Features",
    height=500
)

fig.show()